In [59]:
!pip install gymnasium[atari,accept-rom-license]
import gymnasium as gym
from gymnasium import spaces
!pip install stable-baselines3[extra]
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

In [60]:
from stable_baselines3.common.env_util import make_atari_env
import os
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy


In [61]:
env=gym.make("ALE/Breakout-v5", render_mode='rgb_array')

In [62]:
import ale_py

In [63]:
env.reset()

(array([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        ...,
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]]], dtype=uint8),
 {'lives': 5, 'episode_frame_number': 0, 'frame_number': 0})

In [64]:
env.action_space

Discrete(4)

In [65]:
env.observation_space

Box(0, 255, (210, 160, 3), uint8)

In [66]:
#testing environment
episodes=8
for episode in range(1,episodes+1):
  score=0
  done=False
  obs=env.reset()
  while not done:
    env.render()
    action=env.action_space.sample()
    obs,reward,terminated,truncated,info=env.step(action)
    done = terminated or truncated
    score+=reward
  print(episode, ':', score)

1 : 2.0
2 : 1.0
3 : 1.0
4 : 1.0
5 : 2.0
6 : 2.0
7 : 3.0
8 : 0.0


In the above step it is just learning these randomly and sees how it works. The scores aren't the maximum yet. The highest score so far seems to be 4 above.

In [67]:
#vectorizing environment and training the environment
env = make_atari_env('Breakout', n_envs=1, seed=0)
env = VecFrameStack(env, n_stack=4)

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:520: UserWarning: WARN: Using the latest versioned environment `Breakout-v4` instead of the unversioned environment `Breakout`.
  logger.warn(


Vectorizing the environment, particularly with multiple environments, allows one to train the agent faster by training in parallel.

In [68]:
env.reset()

array([[[[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],

        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],

        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],

        ...,

        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],

        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]],

        [[0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         ...,
         [0, 0, 0, 0],
         [0, 0, 0, 0],
         [0, 0, 0, 0]]]], dtype=uint8)

In [69]:
log_path=os.path.join('Training','Logs')
model=A2C('CnnPolicy', env)
model.learn(total_timesteps=10000)

In [70]:
a2c_path=os.path.join('Training','Saved Models','A2C_Breakout_Model')
model.save(a2c_path)

In [71]:
#evaluating and testing
evaluate_policy(model, env, n_eval_episodes=10, render=True)
#remember: we can only evalute one environment at a time

(np.float64(2.2), np.float64(0.39999999999999997))

In [72]:
env=make_atari_env('Breakout',n_envs=1,seed=0)

In [73]:
env=VecFrameStack(env,n_stack=4)

In [74]:
evaluate_policy(model, env, n_eval_episodes=10, render=True)

(np.float64(2.3), np.float64(0.45825756949558405))

np.float64(2.3): This is the mean reward achieved by your trained A2C model over the 10 evaluation episodes.

np.float64(0.458...): This is the standard deviation of the reward across those 10 evaluation episodes. A higher standard deviation indicates more variability in the scores per episode, while a lower one means the scores were more consistent.

Continue training the A2C model for 1,000,000 timesteps, save the updated model, and then evaluate its performance over 10 episodes for better results.